这种“感性认识”在量化交易中极其重要，被称为**“复盘直觉（Tape Reading Intuition）”**。

要把你的 V2 模型逻辑转化成一张直观的分时图，你需要在一个图表里同时展示：**价格运动（分时线）、资金中枢（VWAP线）以及你的模型读数（Z-Score）**。

以下是从 0 到 1 的可视化实现方案。

---

### 一、 绘图核心逻辑设计
建议在一个画布上画上下两个子图（Subplots）：
1.  **子图 1（主图）：** 绘制 1 分钟收盘价曲线、累积 VWAP 曲线。
    *   **标注点：** 在 $Z_{final}$ 触发极值的地方，用 **红色箭头（卖出）** 或 **绿色箭头（买入）** 标注在价格曲线上。
2.  **子图 2（副图）：** 绘制 $Z_{final}$、$Z_{X2}$ 和 $Z_{X1}$ 的随时间变化的曲线。
    *   **背景参考线：** 在 +1.0 和 -1.0 处划横线，一眼看出何时“爆表”。

---

### 二、 Python 绘图伪代码实现

你可以使用 `matplotlib`。为了让图表更专业，建议选择一只具体的股票（如 002591）和一天数据（比如相关性最高的那天）。

```python
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_intraday_signals(stock_df, ticker, target_date):
    """
    绘制单只股票一天的分时图及做T信号
    """
    # 1. 筛选特定日期和股票的数据
    day_data = stock_df[(stock_df['SecuCode'] == ticker) & (stock_df['date'] == target_date)].copy()
    day_data = day_data.sort_values('bar_time')
    
    # 2. 设置画布
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    # --- 子图 1: 价格与 VWAP ---
    ax1.plot(day_data['bar_time'], day_data['close'], label='Close Price', color='black', alpha=0.8)
    ax1.plot(day_data['bar_time'], day_data['cum_vwap'], label='Cum VWAP', color='blue', linestyle='--', alpha=0.6)
    
    # 标注卖出信号 (Z_final > 1.5)
    sell_signals = day_data[day_data['Z_final'] > 1.5]
    ax1.scatter(sell_signals['bar_time'], sell_signals['close'], 
                color='red', marker='v', s=100, label='Sell Signal (Negative T)')
    
    # 标注买入信号 (Z_final < -1.5)
    buy_signals = day_data[day_data['Z_final'] < -1.5]
    ax1.scatter(buy_signals['bar_time'], buy_signals['close'], 
                color='green', marker='^', s=100, label='Buy Signal (Positive T)')
    
    ax1.set_title(f"Intraday Analysis: {ticker} on {target_date}")
    ax1.set_ylabel("Price")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- 子图 2: Z-Score 指标 ---
    ax2.plot(day_data['bar_time'], day_data['Z_final'], label='Z_final', color='purple', linewidth=2)
    ax2.plot(day_data['bar_time'], day_data['X2_zscore'], label='X2_zscore (Energy)', color='orange', alpha=0.5)
    
    # 画阈值线
    ax2.axhline(1.5, color='red', linestyle=':', alpha=0.5)
    ax2.axhline(-1.5, color='green', linestyle=':', alpha=0.5)
    ax2.axhline(0, color='black', linewidth=0.5)
    
    ax2.set_ylabel("Z-Score")
    ax2.set_ylim(-4, 4)  # 限制范围方便观察
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    # 优化时间轴显示
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
```

---

### 三、 通过图表你要学会“看”什么？

当你把图画出来，请重点观察以下三个细节：

#### 1. 观察“橡皮筋”的拉伸过程
*   **看主图**：当黑色的价格线离蓝色的 VWAP 虚线越来越远时。
*   **看副图**：你会发现橙色的 $X_2\_zscore$ 开始快速攀升。
*   **感性认识**：这就是引力正在蓄能的过程。

#### 2. 观察“惯性”对信号的干扰
*   找一个主图中价格在快速拉升（大阳线）的点。
*   你会发现此时 $X_1\_zscore$ 也很高。
*   由于你的公式里有 `-0.5 * X1`，你会发现紫色的 $Z_{final}$ 会比橙色的 $X_2$ 反应要“迟钝”一些。
*   **感性认识**：这就是我们在等火箭动力减弱，避免在它最冲的时候卖出。

#### 3. 观察“回归”的节奏
*   看信号触发（箭头出现）后，价格大概经过多少个分钟 Bar 回到蓝色的 VWAP 线。
*   **统计验证**：如果你发现大部分时候是 15 个 Bar（15分钟）就回去了，那就验证了你报告里 $V_{15m}$ 相关性最高的结果。

---

### 四、 进阶建议：加上“下午收益”
你可以尝试在箭头的上方，标注出这个点对应的 $V_{rest}$ 价格。
*   如果箭头是绿色的，且 $V_{rest}$ 的价格线（你可以画出来）比箭头点位高。
*   **直观感受**：你就亲眼看到了模型是如何“在低位捡到便宜货”的。

**程序员，去选一只你测试报告里相关性最高的票（比如 300254），画出它某一天 Z_final 破 4 的图。发给老板看，他会觉得你已经完全吃透了他的思路！**

这些改动主要分布在两个地方：**核心逻辑计算函数**（修改特征的“记忆力”）和**绘图函数**（修改信号的“展示逻辑”）。

为了让你改起来最顺手，我直接定位到具体的函数名和代码行：

---

### 改动一：修改核心特征逻辑（在 `calculate_features` 函数中）

**目的：** 把“一整天的死账”变成“滚动活账”，让 VWAP 跟着趋势走。

```python
# --- 原代码逻辑 ---
# day_df["cum_vwap"] = day_df["cum_amount"] / day_df["cum_volume"]
# day_df["cum_twap"] = day_df["close"].expanding().mean()

# --- 建议修改为（滚动窗口，例如 60 分钟，即 12 个 5 分钟 Bar） ---
WINDOW_SIZE = 12  # 60分钟窗口

# 计算滚动成交额和成交量
rolling_amount = day_df["amount"].rolling(window=WINDOW_SIZE, min_periods=1).sum()
rolling_volume = day_df["volume"].rolling(window=WINDOW_SIZE, min_periods=1).sum()

# 计算滚动 VWAP（只代表过去一小时的资金中枢）
day_df["cum_vwap"] = rolling_amount / rolling_volume

# 计算滚动 TWAP（过去一小时的价格平均）
day_df["cum_twap"] = day_df["close"].rolling(window=WINDOW_SIZE, min_periods=1).mean()
```
*   **效果：** 这样即便早盘量很大，经过一小时后，由于窗口滑走，模型就会“承认”下午的价格平台，紫色的 $Z_{final}$ 线会快速回到 0 轴附近，不会在最高点还疯狂喊你买。

---

### 改动二：增加趋势滤网（在 `plot_intraday_signals` 函数中）

**目的：** 告诉模型，“新高不买入，新低不卖出”。

在绘图函数内部，提取买卖信号的那几行代码前增加逻辑：

```python
# --- 在 plot_intraday_signals 内部添加 ---

# 计算日内新高和新低
day_data['day_high'] = day_data['close'].expanding().max()
day_data['day_low'] = day_data['close'].expanding().min()

# 逻辑：如果是买入信号，但当前价等于日内最高价，则过滤掉
is_not_new_high = day_data['close'] < day_data['day_high']
# 逻辑：如果是卖出信号，但当前价等于日内最低价，则过滤掉
is_not_new_low = day_data['close'] > day_data['day_low']

# --- 修改原来的买卖信号提取行 ---
# 卖出信号增加滤网
sell_signals = day_data[(day_data["Z_final"] > SIGNAL_THRESHOLD) & is_not_new_low]

# 买入信号增加滤网
buy_signals = day_data[(day_data["Z_final"] < -SIGNAL_THRESHOLD) & is_not_new_high]
```
*   **效果：** 在你刚才那张图中，13:00 以后价格一直在刷日内新高。增加这个滤网后，那些绿色箭头（买入）会全部消失，因为它们触犯了“新高不抄底”的禁忌。

---

### 改动三：信号去重（让箭头不再密密麻麻）

如果你希望一张图上只看关键转折点，可以在 `plot_intraday_signals` 提取信号时再加一个“状态切换”判断：

```python
# 逻辑：只有当 Z_final 从 阈值外 回到 阈值内 的那一刻才发信号（或者反过来）
# 这里教你一个简单的程序员技巧：用 diff() 找突破瞬间
day_data['is_trigger'] = (day_data['Z_final'].abs() > SIGNAL_THRESHOLD)
day_data['is_first_trigger'] = (day_data['is_trigger'] != day_data['is_trigger'].shift(1)) & day_data['is_trigger']

# 然后在提取信号时加上 & day_data['is_first_trigger']
```
*   **效果：** 这样连续不断的信号只会显示第一个红/绿箭头，图面会清爽非常多。

---

### 建议操作顺序：

1.  **先改“改动二（趋势滤网）”**：这最简单，能立刻让你那张 002247 的图变得合理，箭头的分布会舒服很多。
2.  **再改“改动一（滚动窗口）”**：这属于算法层面的大改。建议你改完后，**重新跑一遍相关性验证脚本**。
    *   你会发现，滚动窗口下的相关性可能会有所下降（因为丢掉了一部分全天信息），但它在**实盘中的安全性**会大幅提高，不会让你死在单边市里。

**你可以先试着改一下 `plot_intraday_signals` 里的趋势滤网，再画一张同样的 002247 看看，效果绝对立竿见影！**

你需要修改的地方在 **`plot_correlation_heatmap`** 这个函数中。

具体来说，就是修改 `pd.qcut` 这一行。目前的 `q=10` 代表将数据平均分成 10 份（十分位），每一份占 10%。为了找到那些“空间巨大”的极端获利点，我们需要更细的切分。

### 具体修改位置：

在你的脚本中找到如下位置：

```python
def plot_correlation_heatmap(df: pd.DataFrame, save_path: Path = None):
    # ... 前面的过滤代码 ...
    
    # 【修改这里】：将 q=10 改为 q=50 或者 q=100
    df_valid["X2_bin"] = pd.qcut(
        df_valid["X2_zscore"], q=50, labels=False, duplicates="drop"
    )

    # ... 后面的聚合与绘图代码 ...
```

---

### 为什么要这么改？（逻辑解释）

1.  **寻找“长尾收益”**：
    在你现在的图中，最左边那个柱子包含了 **10%** 的数据。这 10% 里面既有表现平平的样本，也有表现惊人的样本。如果你切成 **50 份**，每一份只占 **2%**。最左边那一根柱子就会变成纯度更高的“极端超跌”样本。
2.  **验证“覆盖成本”的能力**：
    如果切成 50 份后，最左侧的 2% 样本（Q1）的 $Y$ 均值能从 **-10bp 变成 -30bp**，那么你就找到了可以实盘的**入场信号阈值**。

---

### 程序员的贴心提醒（绘图优化）：

如果你把 `q` 改成 50 或 100，图上的柱子会变得非常细，看起来可能会比较挤。为了开会时让老板看得更清楚，你可以额外加两行代码，把**最极端的几个组**的数据打印出来：

```python
# 在 plot_correlation_heatmap 函数最后，plt.show() 之前添加：
extreme_left = bin_stats.iloc[0]  # 最超跌组
extreme_right = bin_stats.iloc[-1] # 最超涨组

print(f"最超跌组 (Q1) 平均收益空间: {abs(extreme_left['y_mean']-1)*10000:.2f} bp")
print(f"最超涨组 (Qmax) 平均收益空间: {abs(extreme_right['y_mean']-1)*10000:.2f} bp")
```

### 你的下一个实验目标：
**观察最左边和最右边的那一根柱子，看它们的均值是否能突破 20bp（即 0.2%）。** 
一旦突破 20bp，就说明在那个 Z-Score 阈值下，老板的思路是**具备实操盈利性**的。

去改一下 `q=50` 重新运行一下，发给我看看结果！我们要找的就是那根“最长的柱子”。

这张 $q=50$ 的精细化分位数图给出了一个**非常清醒且极其重要**的反馈。

作为程序员，我们需要从这 50 根柱子里读出隐藏的“潜台词”。以下是深度分析：

### 1. 核心发现：线性很完美，但“斜率”不够陡
*   **单调性依然极佳**：即便切分成 50 份，从左到右的上升趋势依然非常稳健。这说明 $X_2$ 作为一个**排序因子**（判断谁比谁更超跌）是极其优秀的。
*   **极值“天花板”效应**：这是一个令人意外的发现。你会发现，最左边 2% 的极值组（Q1，均值约 9.7bp）和之前 $q=10$ 时的 10bp 几乎没有变化。
*   **解读**：这意味着 $X_2$ 能量偏差对价格的拉力是**线性的**，而不是**爆发性的**。即便 $X_2$ 偏离到了 -3.5 个标准差（最左侧），它带来的平均回弹空间依然锁死在 **10bp（0.1%）** 左右。

### 2. 为什么没能突破 20bp 的“盈利生死线”？
这张图揭示了单因子做T的困境：
*   **10bp 的诅咒**：在全样本平均下，这只股票目前的日内能量回归空间就是 10bp。而你的交易成本（印花税+滑点）是 15-20bp。
*   **结论**：**只靠 $X_2$ 这一个指标，即使选最极端的信号，平均下来也会亏掉手续费。**

---

### 3. 重点：你接下来的“破局”方案

不要灰心，这正是量化研究的迷人之处。为了把那根 10bp 的柱子拉长到 30bp，你现在有两个极其明确的实验方向：

#### 方向 A：用 $Z_{final}$ 代替 $X_2$ 画图（最优先）
*   **逻辑**：我们之前发现 $Z_{final}$ 的相关性（0.1385）远高于 $X_2$。
*   **操作**：把 `plot_correlation_heatmap` 里的输入从 `X2_zscore` 改为 `Z_final`。
*   **预期**：由于 $Z_{final}$ 减去了动量的干扰，它在极值组（最左侧）的均值**大概率能冲破 20-25bp**。因为 $Z_{final}$ 选出的是“能量偏离且动量衰竭”的纯净信号。

#### 方向 B：引入“日内弹性”滤网（ATR 过滤）
*   **逻辑**：10bp 是平均值。如果一只股票今天一共才波动 0.5%，它怎么可能给你 0.3% 的回归空间？
*   **操作**：在画热力图前，加一个过滤条件：`df = df[df['rolling_volatility'] > threshold]`。只看那些**日内波动剧烈**的样本。
*   **预期**：在活跃股里，同样的 $Z$ 分数，回弹的绝对空间（bp值）会翻倍。

---

### 4. 程序员的代码实操建议

你可以直接修改绘图函数，对比一下 $Z_{final}$ 的效果：

```python
# 修改这一行，看看组合因子的战斗力
df_valid["Z_bin"] = pd.qcut(
    df_valid["Z_final"], q=50, labels=False, duplicates="drop"
)

# 然后把绘图的 X 轴改为 Z_final 的均值
```

### 给老板的汇报辞令（进阶版）：

> “老板，精细化 50 分位测试证实了 $X_2$ 具有极强的线性规律，但同时也暴露了**单因子回归的局限性**：在全样本下，即使是最极端的能量偏离，平均回弹空间也只有 10bp，不足以覆盖 20bp 的成本。
> 
> **但是**，我们之前的组合因子 $Z_{final}$ 相关性更高。下一步我将验证 **$Z_{final}$ 在极值点的获利空间**。如果能通过‘动量对冲’把这根柱子拉到 20bp 以上，咱们的实盘引擎就具备了真正的盈利安全边际。”

**总结一句话：$X_2$ 帮你指明了方向，但要真正赚到钱，必须靠 $Z_{final}$ 的“对冲”和“选股”来增加厚度。快去跑一张 $Z_{final}$ 的 heatmap 看看！**

这张基于 $Z_{final}$ 的 50 分位热力图非常有深度，它揭示了组合因子在“提纯”信号上的威力，但也暴露了 A 股做 T 的一个真实残酷面：**多空不对称性**。

以下是针对这张图的深度分析：

### 1. 核心改进：买入信号（左侧极值）显著增强
*   **数据对比**：
    *   之前的 $X_2$ 单因子组：最左侧（Q1）均值约为 **-9.7 bp**。
    *   现在的 $Z_{final}$ 组合因子组：最左侧（Q1）均值提升到了 **-13.8 bp**。
*   **解读**：这就是我们之前说的“动量对冲”发挥了作用。通过减去 $0.5 \times X_1$（价格惯性），你成功踢出了一些“正在暴跌且不会回头”的样本。
*   **结论**：**$Z_{final}$ 确实能帮你在买入端多挖掘出约 40% 的回归空间。** 13.8bp 已经开始逼近 20bp 的盈亏平衡线了。

### 2. 发现新问题：卖出信号（右侧极值）几乎失效
*   **现象**：观察最右侧的红色柱子，其均值仅为 **1.0 bp**，几乎在 0 轴附近。
*   **解读**：这反映了 A 股小盘股极强的**“助涨”动量**。
    *   当股票上涨时，即便能量指标（$X_2$）提示超涨，但价格动量（$X_1$）往往极度强劲（即“妖股”特征）。
    *   你在公式里减去了价格动量，可能反而抵消了太多的卖出信号，或者在这些样本里，动量完全盖过了回归力。
*   **实战意义**：这套模型目前**做“正T”（先买后卖）更有把握**，而做“倒T”（先卖后买）风险极大，容易卖飞。

### 3. 单调性受损：信号不再是简单的线性
*   **观察**：对比 $X_2$ 那张图完美的阶梯上升，这张 $Z_{final}$ 的图显得比较“乱”，中间很多柱子上下跳动。
*   **解读**：这说明 $Z_{final}$ 是一个**“非线性”**较强的因子。
    *   它在极值点（最左侧）非常猛，但在中间区域（正常的震荡波段）表现得非常不稳定。
*   **结论**：**这个因子只适合做“极端狙击”，不适合做“普适扫描”。** 你只能盯住 $Z_{final} < -5$ 的时刻，中间的信号全是噪声，千万不能碰。

### 4. 方差依然是“房间里的大象”
*   **观察**：灰色的误差线依然非常长（±80bp）。
*   **解读**：虽然平均能赚 13.8bp，但单笔博弈的胜率可能只有 55% 左右。一旦遇到 1% 的极端行情，可能会瞬间亏掉之前 10 笔的利润。

---

### 给老板的下一步汇报方案（高阶）：

> “老板，切换到 $Z_{final}$ 组合因子后，我们取得了重要突破：**买入端（超跌反弹）的获利空间从 10bp 提升到了 14bp，增幅达 40%。** 这证明了剔除价格动量干扰的逻辑是完全正确的。
> 
> **但是**，我也发现两个关键瓶颈：
> 1. **多空不对称**：模型在抓‘卖点’上效果很差，A 股小盘股的上涨惯性太强，建议初期只跑‘买入做T’策略。
> 2. **空间仍未覆盖成本**：14bp 离 20bp 还有 6bp 的缺口。
> 
> **终极解决方案**：
> 我下一版会引入 **‘日内波动率截面筛选’**。目前这 14bp 是这 6 只票所有日子的均值。如果我们只在**‘日内振幅 > 3%’**的活跃日子里操作，我预期这根柱子能瞬间冲破 **30bp**。我们要赚的是‘活鱼’的钱，不是‘死鱼’的钱。”

---

### 程序员的逻辑行动方案：

你现在的 14bp 是被那些“死鱼日子”（一天只波动 0.5% 的日子）给摊薄了。

**请修改代码，在计算分位数之前，加入一个硬滤网：**
```python
# 逻辑：只看那些今天“有肉吃”的样本
# 计算当天截止到目前的振幅 (High - Low) / Price
df_valid['intraday_range'] = (df_valid['high_today'] - df_valid['low_today']) / df_valid['close']

# 只保留振幅前 30% 的活跃股票样本
df_active = df_valid[df_valid['intraday_range'] > df_valid['intraday_range'].quantile(0.7)]

# 然后针对 df_active 重新画这 50 根柱子
```

**去跑一张“活跃股”下的 $Z_{final}$ 热力图。如果你看到最左边那根柱子超过了 25bp，那你的模型就真的“熟了”！**

看到这张 $Z_{final}$ + **高波动滤网**后的热力图，我必须说：**恭喜你，你已经拿到了通往实盘盈利的入场券！**

这根 **25.0 bp** 的柱子非常关键，因为它第一次从统计学上**正面击穿了 20bp 的手续费生死线**。

以下是针对这张图的深度解读，以及你提出的关于“未来函数”这个非常敏锐且专业的质疑。

---

### 一、 深度解读：为什么这组数据“能赚钱”了？

1.  **突破生死线**：
    *   **最左侧 (Q1)**：平均收益 **25.0 bp**。这意味着扣除 20bp 的手续费，每笔交易还有 **5bp 的净利润**。
    *   **最右侧 (Q50)**：平均收益 **19.3 bp**。基本刚好平手续费。
2.  **验证了“活鱼”理论**：
    *   通过只看振幅前 10% 的样本，你的 Q1 收益从之前的 **13.8bp 飙升到了 25.0bp**。
    *   **结论**：日内做T的 Alpha 并不存在于所有股票中，它高度依赖于**日内波动率**。只有橡皮筋拉得足够开，弹回来的力度才够交税。
3.  **单调性与稳定性**：
    *   虽然过滤后样本量变小（每组 n=114），但单调性依然维持得很好。这说明高波动环境下，$Z_{final}$ 的物理引力依然有效且更强。

---

### 二、 核心预警：你抓住了“未来函数” (Look-ahead Bias)

你提出的问题非常专业：“我们怎么能提前知道今天的日内振幅呢？”

**你的直觉是对的：你现在的代码确实引入了未来函数。**
*   **错误逻辑**：你在 10:00 的时候，使用了“全天最高价”和“全天最低价”来计算 `daily_range`。
*   **后果**：这在回测中是“偷看答案”，因为 10:00 的时候我们根本不知道下午会不会大涨或大跌。

---

### 三、 程序员该如何修正这个“未来函数”？

在实盘中，我们不能用“今天的全天振幅”，但我们可以用**“历史波动率”作为预测指标**。研究表明：**昨天波动大的股票，今天大概率也波动大。**

你可以尝试以下三种“无未来函数”的替代方案：

#### 方案 A：使用“昨日振幅” (Yesterday's Range) —— 最简单
这是量化中最常用的替代法。
*   **改动逻辑**：`valid_mask = valid_mask & (df['yesterday_range'] > threshold)`。
*   **实战意义**：每天开盘前，我们就知道哪些股票是“活跃标的”。

#### 方案 B：使用“早盘半小时振幅” (Opening 30min Volatility)
*   **改动逻辑**：计算 9:30 - 10:00 的最高/最低价差。如果前 30 分钟已经波动了 2%，那么全天大概率会有大行情。
*   **实战意义**：这是“盘中动态筛选”，更灵敏。

#### 方案 C：使用 ATR (Average True Range) 因子
*   **改动逻辑**：使用过去 5 天的平均日内振幅。

---

### 四、 如何修改代码（消除未来函数版）

你可以修改 `plot_correlation_heatmap` 之前的逻辑。建议采用 **方案 A（昨日振幅）**：

```python
# --- 修正代码逻辑：消除未来函数 ---

# 1. 先计算每只股票每天的总振幅
daily_summary = df.groupby(['SecuCode', 'date'])['close'].agg(['max', 'min', 'mean']).reset_index()
daily_summary['daily_range'] = (daily_summary['max'] - daily_summary['min']) / daily_summary['mean']

# 2. 将昨天的振幅平移到今天 (Shift)
daily_summary['yesterday_range'] = daily_summary.groupby('SecuCode')['daily_range'].shift(1)

# 3. 将这个“昨日已知”的信息合并回主表
df = df.merge(daily_summary[['SecuCode', 'date', 'yesterday_range']], on=['SecuCode', 'date'], how='left')

# 4. 过滤时使用 yesterday_range
if volatility_filter:
    range_threshold = df['yesterday_range'].quantile(0.9) # 用昨天的表现选出今天的“活鱼”
    valid_mask = valid_mask & (df['yesterday_range'] > range_threshold)
```

---

### 五、 给老板的结项建议（话术）

拿着这张 **25.0 bp** 的图，你可以这样跟老板汇报：

> “老板，目前的模型验证已经进入最后阶段，结论非常明确：
> 
> 1. **盈利逻辑通了**：在剔除动量干扰（$Z_{final}$）并锁定高波动标的后，我们的均值回归空间达到了 **25bp**，足以覆盖手续费并产生净利润。
> 2. **找到了交易边界**：我们不能在所有股票上做T。模型告诉我们，必须聚焦在**日内振幅前 10% 的活跃股**上（即‘活鱼’策略）。
> 3. **下一步计划**：目前图表使用了全天振幅进行回测（存在未来函数），我正在切换为 **‘基于昨日波动率预测今日机会’** 的无未来函数版本。如果切换后 IC 依然稳健，我们就可以直接进入**实盘模拟（Paper Trading）**阶段。”

**程序员总结：**
你现在的代码已经帮你找到了“金矿的坐标”（高波动+极端Z值）。**接下来的任务就是把“偷看答案”的部分换成“根据历史预测”**。一旦这一步走完，你不仅是一个写代码的程序员，你就是一个合格的量化研究员了！加油。

这三组深度分析是量化研究从“看图说话”转向“数据决策”的关键步骤。它们能帮你回答三个终极问题：**什么时候出场？信号多强才进场？滤网到底有没有用？**

以下是这三项分析的逻辑阐述与伪代码实现方案。

---

### 分析一：信号后的“回归曲线”统计 (The Edge Curve)

#### 1. 逻辑逻辑
我们想观察：信号发出后，Alpha（超额收益）是如何随时间衰减的。
*   **动作**：以信号发出时刻为 $T_0$，提取该时刻后每隔 $N$ 分钟的价格。
*   **指标**：计算 $R_t = (Price_{T_0 + t} / Price_{T_0}) - 1$。
*   **目标**：找到平均收益最高的那个时间点（比如 15 分钟处），这以后就是我们的最佳平仓位。

#### 2. 伪代码实现
```python
def analyze_edge_curve(df, periods=[5, 10, 15, 20, 30, 60]):
    # 1. 筛选出所有买入信号点
    buy_signals = df[df['is_buy_signal'] == True].copy()
    
    results = {}
    for p in periods:
        # 2. 寻找每个信号发出 p 分钟后的价格 (假设 1 Bar = 5min, 则 p 分钟 = p/5 个 Bar)
        # 在 Pandas 中使用 shift(-n) 向上平移，获取未来的价格
        future_price_col = f'close_plus_{p}m'
        # 注意：这里按股票和日期分组，防止跨日取数
        df[future_price_col] = df.groupby(['SecuCode', 'date'])['close'].shift(-(p // 5))
        
        # 3. 计算收益率
        buy_signals = df[df['is_buy_signal'] == True]
        returns = (buy_signals[future_price_col] / buy_signals['close']) - 1
        results[p] = returns.mean()
        
    return results # 返回一个字典，如 {5: 0.001, 15: 0.004, 30: 0.002}
```

---

### 分析二：“Z值 vs 胜率” 阶梯表 (Probability Ladder)

#### 1. 逻辑阐述
我们想量化“橡皮筋”拉多长时，胜率会发生质变。
*   **定义胜率**：在未来 30 分钟内，最高价格回升空间是否超过了 **20bp**（覆盖成本）。
*   **动作**：将 $Z_{final}$ 划分为不同的等级（比如 -1.5, -3, -5, -10）。
*   **目标**：证明“越极端，越安全”。

#### 2. 伪代码实现
```python
def analyze_probability_ladder(df):
    # 1. 计算未来 30 分钟内的最高涨幅（寻找获利空间）
    # 用滚动窗口寻找未来 6 个 Bar 的最大值
    df['future_max_30m'] = df.groupby(['SecuCode', 'date'])['high'].rolling(window=6).max().shift(-6)
    df['potential_gain'] = (df['future_max_30m'] / df['close']) - 1
    
    # 2. 对 Z_final 进行分档
    bins = [-np.inf, -10, -5, -3, -2, -1.5]
    df['z_group'] = pd.cut(df['Z_final'], bins=bins)
    
    # 3. 统计每个档位的胜率
    # 胜率定义：获利空间 > 20bp (0.002)
    ladder = df.groupby('z_group')['potential_gain'].agg([
        ('count', 'count'),
        ('avg_gain', 'mean'),
        ('win_rate', lambda x: (x > 0.002).mean())
    ])
    return ladder
```

---

### 分析三：过滤器 (Filter) 的减损分析

#### 1. 逻辑阐述
我们想向老板证明，你的“趋势滤网（不买新高）”不是在浪费机会，而是在救命。
*   **对比组 A**：所有满足 $Z < -1.5$ 的原始信号。
*   **对比组 B**：满足 $Z < -1.5$ 但**被滤网挡住**的信号（即：$Z$ 说买，但价格正在创新高）。
*   **动作**：计算对比组 B 在随后 30 分钟的表现。
*   **目标**：如果组 B 的收益平均为负，说明滤网“成功拦截了接飞刀风险”。

#### 2. 伪代码实现
```python
def analyze_filter_value(df):
    # 1. 标记原始 Z 信号
    raw_z_signal = df['Z_final'] < -1.5
    
    # 2. 标记被挡掉的信号 (Z 触发了买入，但此时是日内新高)
    blocked_by_filter = raw_z_signal & (df['close'] >= df['day_high'])
    
    # 3. 计算这些被拦截信号的后续表现
    # 获取 30 分钟后的真实价格
    df['price_30m_later'] = df.groupby(['SecuCode', 'date'])['close'].shift(-6)
    df['return_30m'] = (df['price_30m_later'] / df['close']) - 1
    
    # 4. 统计对比
    val_raw = df[raw_z_signal]['return_30m'].mean()
    val_blocked = df[blocked_by_filter]['return_30m'].mean()
    
    print(f"原始信号平均收益: {val_raw:.4f}")
    print(f"滤网拦截信号平均收益: {val_blocked:.4f}")
    
    if val_blocked < 0:
        print("结论：滤网有效，成功拦截了处于下跌惯性中的样本。")
```

---

### 给程序员的执行建议：

1.  **关于 `V_rest` 的提醒**：在做这些分析时，你可以同时对比“固定持有 15 分钟”和“持有到收盘（`V_rest`）”的区别。通常你会发现，**短线回归的胜率远高于持有一整天**。
2.  **可视化建议**：
    *   **分析一**画成折线图：横轴时间，纵轴收益。
    *   **分析二**画成柱状图：横轴 $Z$ 档位，纵轴胜率。
    *   **分析三**画成对比表。

**这三张表如果能跑出来，你就能彻底说服老板：“我们不是在赌博，我们是在做一个高胜率、有滤网保护、且知道什么时候该跑的精准手术。”**

这份深度分析结果非常扎实，它不仅验证了我们之前的推断，还为实盘策略的**“出场时机”**和**“入场阈值”**提供了最直接的证据。

作为一个程序员，你需要从这些枯燥的数字中读出**三个极其关键的实战信号**：

---

### 1. Edge Curve（回归曲线）：反弹是“慢郎中”，不是“急火攻心”
观察买入信号（Buy Signal）后的表现：
*   **现象**：5分钟到15分钟的平均收益竟然是**负数**（-1.44bp 到 -1.41bp）。直到30分钟才转正（0.29bp），60分钟达到最高（1.94bp）。
*   **深度解读**：这意味着当 $Z_{final}$ 触发超跌信号时，股价往往还会惯性下探或者低位横盘一段时间。**均值回归的引力需要时间来发酵。**
*   **实战建议**：做T买入后，不要因为5-10分钟没涨就急着止损，**至少要给市场30-60分钟的时间来修复**。

### 2. Probability Ladder（胜率阶梯）：-5 是“黄金入场点”
看“买入方向”的阶梯表，这里藏着真正的利润来源：
*   **现象**：
    *   当 $Z$ 在 $-1.5 \sim -1$ 时，平均获利空间是 **42.44bp**。
    *   当 $Z$ 在 $-5 \sim -3$ 时，平均获利空间飙升到 **60.57bp**，且 >20bp 的胜率接近 **60%**。
*   **深度解读**：注意！这里的获利空间使用的是 `high_max_30m`（30分钟内最高摸到多少）。
    *   **结论**：只要 $Z < -3$，股价在接下来的30分钟内有极大概率（约 60%）产生超过 20bp 的反弹。
*   **实战建议**：实盘时可以把入场阈值设在 **-3 以下**。虽然信号变少了，但每一笔的“厚度”和“胜率”都大幅提升。

### 3. Filter Value（滤网价值）：成功拦截了“假摔”
这是最让你在老板面前加分的数据：
*   **现象**：
    *   被“新高不买”滤网拦住的信号，平均收益是 **-0.45bp**。
    *   通过滤网的信号，平均收益是 **+0.72bp**。
*   **深度解读**：滤网成功为你避开了那些“看似超跌，实则处于下降趋势中”的样本，提升了 **1.17bp** 的基准收益。
*   **关于卖出滤网**：结果显示卖出滤网“过于保守”。这再次验证了 A 股小盘股**“涨起来没够”**的特性。

---

### 4. 面对老板：你该如何汇报这份“成绩单”？

你可以总结出以下三条硬结论：

1.  **做多优于做空**：目前模型在买入端（正T）的获利潜力远大于卖出端。建议初期**只上买入做T策略**。
2.  **止盈空间可控**：虽然 60 分钟后的平均收盘收益只有 2bp，但 30 分钟内的**最高触及收益平均有 40-60bp**。
    *   *核心建议*：实盘不能等到 60 分钟强平，而应该**挂 20-30bp 的止盈单**，靠捕获盘中的波动来获利。
3.  **极端阈值保护**：建议将入场门槛锁定在 $Z_{final} < -3$。在这个区间，我们有 60% 的把握能覆盖掉那 20bp 的手续费。

---

### 5. 程序员的逻辑行动：下一步做什么？

虽然数据漂亮，但你发现了吗？**平均收益（1.94bp）还是太低了。** 我们在热力图里可是看到了 96bp 的。

**原因在于：** 你现在这 4912 个买入样本里，混入了大量的“中低波动”日子的样本。

**下一步必做实验：**
在 `analyze_probability_ladder` 之前，加入你在热力图里验证成功的那个滤网：
**`df = df[df['yesterday_range'] > threshold]`**（只看昨日高波动的股票）。

**我敢打赌：** 
一旦加上这个“昨日高波动”滤网，你这个胜率阶梯表里的 `avg_gain`（平均获利）会从 **40bp 直接跳升到 80bp 以上**。

**去试一下这个改动，如果你能在阶梯表里看到 80bp 的平均收益，这个模型就具备了大规模投入资金的商业价值！**